# TP 2 — Tokeniser un corpus

## Corpus du semestre

Le corpus réunit neuf œuvres françaises du XIXe siècle, toutes dans le domaine public :

| Auteur | Œuvres |
|---|---|
| Gustave Flaubert | *Madame Bovary* ; *L'Éducation sentimentale* ; *Bouvard et Pécuchet* |
| Guy de Maupassant | *Une vie* ; *Bel-Ami* ; *Pierre et Jean* |
| Émile Zola | *Germinal* ; *L'Assommoir* ; *Au Bonheur des Dames* |

Ce corpus nous servira pour l'ensemble de nos TD.

Pour ce TP, nous utiliserons [*Madame Bovary*, eBook 14155](https://www.gutenberg.org/ebooks/14155) de Project Gutenberg.

## Bibliothèques utilisées

Les modules `pathlib`, `urllib` et `re` appartiennent à la bibliothèque standard de Python : ils n'ont pas besoin d'être installés.

- `pathlib` manipule les chemins et les fichiers ;
- `urllib` télécharge le texte lors de la première exécution ;
- `re` applique des expressions régulières.

La bibliothèque `transformers` fournit une interface commune pour charger les tokenizers associés à de nombreux modèles. `AutoTokenizer` sélectionne automatiquement la classe adaptée à partir du nom du modèle. Les tokenizers peuvent segmenter le texte, convertir les tokens en identifiants, ajouter les tokens spéciaux et préparer la troncation ou le padding.

Documentation : [tokenizers de Transformers](https://huggingface.co/docs/transformers/main_classes/tokenizer), [`AutoTokenizer`](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoTokenizer), [algorithmes de tokenisation](https://huggingface.co/docs/transformers/tokenizer_summary).

## Installer Transformers

Dans un terminal, après activation de l'environnement utilisé par Jupyter :

```bash
python -m pip install transformers sentencepiece
```

Depuis le notebook :

```python
%pip install transformers sentencepiece
```

`sentencepiece` est nécessaire à certains tokenizers. Nous ne chargeons pas les modèles neuronaux dans ce TP : il n'est donc pas nécessaire d'installer PyTorch. La première utilisation de `from_pretrained()` télécharge les fichiers du tokenizer, qui sont ensuite conservés dans le cache local.

Documentation : [installation de Transformers](https://huggingface.co/docs/transformers/installation).

## 1. Charger le texte

La cellule suivante télécharge le texte seulement si le fichier n'existe pas déjà. Une fois le fichier enregistré, le TP peut être repris sans nouveau téléchargement.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen

url = "https://www.gutenberg.org/cache/epub/14155/pg14155.txt"
fichier = Path("corpus/madame_bovary_gutenberg_14155.txt")

if not fichier.exists():
    fichier.parent.mkdir(parents=True, exist_ok=True)
    requete = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(requete) as reponse:
        contenu = reponse.read().decode("utf-8-sig")
    fichier.write_text(contenu, encoding="utf-8")

texte_brut = fichier.read_text(encoding="utf-8")
print(f"{len(texte_brut):,} caractères chargés")

Le fichier contient aussi une notice et la licence de Project Gutenberg. Elles ne font pas partie du roman et doivent être retirées avant de calculer des statistiques.

In [ ]:
marque_debut = "*** START OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"
marque_fin = "*** END OF THE PROJECT GUTENBERG EBOOK MADAME BOVARY ***"

texte = texte_brut.split(marque_debut, 1)[1]
texte = texte.split(marque_fin, 1)[0].strip()

print(f"{len(texte):,} caractères conservés")
print(texte[:500])

## 2. Première approche : `split()`

Sans argument, `split()` découpe une chaîne sur les espaces, tabulations et retours à la ligne.

In [ ]:
phrase = "Aujourd'hui, l'ami de Jean-Pierre s'est dit : j'aime New York."
tokens_exemple = phrase.split()

print(tokens_exemple)
print(len(tokens_exemple))

La ponctuation reste attachée aux chaînes voisines. Ainsi, `Aujourd'hui,` et `Aujourd'hui` seraient deux types différents.

### Exercice 1

1. Tokenisez le roman avec `split()`. Affichez le nombre de tokens et les quarante premiers tokens.

2. Un `set` ne conserve qu'un exemplaire de chaque valeur. Construisez le vocabulaire du roman, puis affichez le nombre de types.

3. Calculez le TTR en divisant le nombre de types par le nombre de tokens. Recommencez après avoir passé tous les tokens en minuscules avec `.lower()`.

4. Calculez le TTR sur les 1 000, 5 000, 10 000 et 50 000 premiers tokens. Présentez pour chaque taille le nombre de types et le TTR. Que devient le TTR lorsque la longueur augmente ?

Le TTR dépend fortement de la longueur du texte. Deux TTR calculés sur des nombres de tokens très différents ne doivent pas être interprétés comme une comparaison directe de diversité lexicale.

## 3. Deuxième approche : les expressions régulières

`re.findall(motif, texte)` renvoie toutes les portions du texte qui correspondent au motif. Le motif `\w+` repère des suites de caractères alphanumériques, mais sépare les apostrophes et les traits d'union.

In [ ]:
import re

tokens_mots = re.findall(r"\w+", phrase.lower())
print(tokens_mots)

On peut conserver les apostrophes et traits d'union internes avec un motif plus précis :

```python
r"[^\W\d_]+(?:['’-][^\W\d_]+)*"
```

`[^\W\d_]` désigne ici une lettre Unicode. La seconde partie autorise une ou plusieurs séquences composées d'une apostrophe ou d'un trait d'union suivis de lettres. Ce choix traite `aujourd'hui` et `Jean-Pierre` comme un seul token. C'est une décision de tokenisation, pas une vérité linguistique.

In [ ]:
motif_francais = r"[^\W\d_]+(?:['’-][^\W\d_]+)*"
tokens_francais = re.findall(motif_francais, phrase.lower())
print(tokens_francais)

Documentation : [`re`](https://docs.python.org/fr/3/library/re.html), [`re.findall`](https://docs.python.org/fr/3/library/re.html#re.findall).

### Exercice 2

1. Appliquez les deux expressions régulières à la phrase d'exemple. Comparez le nombre et la forme des tokens obtenus.

2. Tokenisez tout le roman en minuscules avec `motif_francais`. Calculez le nombre de tokens, le nombre de types et le TTR.

3. Comparez ce TTR à celui obtenu avec `split()`. Examinez trente éléments présents dans le vocabulaire produit par `split()` mais absents du vocabulaire produit par l'expression régulière.

4. Écrivez une fonction `ttr(tokens)` qui reçoit une liste de tokens et renvoie le rapport types/tokens. Vérifiez-la avec les deux tokenisations.

## 4. Tokenizers préentraînés de Hugging Face

Nous comparerons trois tokenizers :

| Modèle | Procédure | Corpus d'apprentissage |
|---|---|---|
| `camembert-base` | SentencePiece | français |
| `bert-base-multilingual-cased` | WordPiece | multilingue |
| `gpt2` | BPE au niveau des octets | principalement anglais |

Les différences observées proviendront à la fois de l'algorithme, de la taille du vocabulaire et des données sur lesquelles le vocabulaire a été appris.

In [ ]:
from transformers import AutoTokenizer

modeles = {
    "CamemBERT": "camembert-base",
    "BERT multilingue": "bert-base-multilingual-cased",
    "GPT-2": "gpt2",
}

tokenizers = {}
for nom, identifiant in modeles.items():
    print(f"Chargement de {nom}")
    tokenizers[nom] = AutoTokenizer.from_pretrained(identifiant)

`tokenize()` renvoie les tokens sous forme de chaînes. `encode()` renvoie les identifiants numériques et peut ajouter les tokens spéciaux attendus par le modèle. `decode()` effectue l'opération inverse.

In [ ]:
phrase_test = (
    "Aujourd'hui, l'ami de Jean-Pierre s'est dit : "
    "j'aime New York. L'hyperconnectivité bouleverse-t-elle nos usages ?"
)

for nom, tokenizer in tokenizers.items():
    tokens = tokenizer.tokenize(phrase_test)
    print(f"\n{nom} : {len(tokens)} tokens")
    print(tokens)

### Exercice 3

1. Comparez les segmentations de la phrase test. Repérez le traitement des apostrophes, du trait d'union, de `New York` et de `hyperconnectivité`.

2. Faites tokeniser séparément `chat`, `chatons`, `anticonstitutionnellement`, `covidé`, `Jean-Pierre` et `New York` par les trois tokenizers. Présentez les résultats de manière lisible.

3. Pour chaque tokenizer, affichez `vocab_size` et `all_special_tokens`. Les trois vocabulaires ont-ils la même taille ? Les tokens spéciaux ont-ils la même forme ?

4. Encodez la phrase test avec chaque tokenizer en conservant les tokens spéciaux. Affichez les identifiants, reconvertissez-les en tokens avec `convert_ids_to_tokens()`, puis décodez la séquence.

5. Prenez les 1 000 premiers caractères du roman. Pour chaque tokenizer, calculez le nombre de sous-tokens produits. Comparez ce nombre au nombre de tokens obtenu par l'expression régulière sur le même extrait. Quel tokenizer produit la séquence la plus longue ?

## Bilan

Réalisez une analyse commentée des résultats donnés par les différents tokenizer.